# 01 Features: `ml.store_sku_features`

One row per ranged store x SKU x day with what the business knows at the start of the day:
sales and stockouts up to the day before (lags 1/7/14/28, rolling means and spread, a
censoring-corrected rate, stockout shares), channel mix, the region x category trend
(seasonality), and the plan for the day (promotion, price, day of week). The definitions live in
`bigbox_ml/features.py`, shared by training, scoring, the backtest and the endpoint.

Written as a Unity Catalog feature table (Feature Engineering client) with primary key
(store_id, sku_id, date) and `date` as the timeseries key, so training can use point-in-time
lookups. `mode=full` rebuilds every row; `mode=incremental` (the 15-minute refresh) recomputes
the last few days, which late-arriving sales files can still change.

In [ ]:
import datetime as dt
import json
import time

from pyspark.sql import functions as F

dbutils.widgets.text("catalog", "bigbox_fe_bar")
dbutils.widgets.dropdown("mode", "full", ["full", "incremental"])
CAT = dbutils.widgets.get("catalog")
MODE = dbutils.widgets.get("mode")
TABLE = f"{CAT}.ml.store_sku_features"
LATE_DAYS = 4  # POS files can land up to 3 days late (generator config); recompute one day more

from databricks.feature_engineering import FeatureEngineeringClient  # noqa: E402

from bigbox_ml import features  # noqa: E402

fe = FeatureEngineeringClient()
daily = spark.read.table(f"{CAT}.gold.store_sku_daily")
products = spark.read.table(f"{CAT}.silver.products")
exists = spark.catalog.tableExists(TABLE)
print(f"mode={MODE}, feature table exists={exists}")

mode=full, feature table exists=True


In [ ]:
t0 = time.time()
start = None
if MODE == "incremental" and exists:
    last = daily.agg(F.max("date")).first()[0]
    start = str(last - dt.timedelta(days=LATE_DAYS))
feats = features.build(daily, products, start)
if not exists:
    fe.create_table(
        name=TABLE,
        primary_keys=["store_id", "sku_id", "date"],
        timeseries_column="date",
        df=feats,
        description=(
            "Demand-model features per ranged store x SKU x day, as known at the start of the "
            "day (history up to the day before + the day's plan). Source: gold.store_sku_daily, "
            "silver.products. Definitions: src/ml/bigbox_ml/features.py."
        ),
    )
else:
    fe.write_table(name=TABLE, df=feats, mode="merge")
print(f"written in {time.time() - t0:.0f} s (recomputed from {start or 'the first day'})")

written in 76 s (recomputed from the first day)


In [ ]:
for col, doc in features.DOCS.items():
    spark.sql(
        f"ALTER TABLE {TABLE} ALTER COLUMN {col} COMMENT '{doc.replace(chr(39), chr(39) * 2)}'"
    )
spark.sql(f"ALTER TABLE {TABLE} SET TAGS ('domain' = 'supply_chain', 'sensitivity' = 'internal')")
print(f"{len(features.DOCS)} column comments and table tags set")

28 column comments and table tags set


## What was written

In [ ]:
ft = spark.read.table(TABLE)
summary = ft.agg(
    F.count("*").alias("rows"),
    F.countDistinct("store_id", "sku_id").alias("store_skus"),
    F.min("date").alias("first_date"),
    F.max("date").alias("last_date"),
).first()
print(summary.asDict())
nulls = ft.select(
    [F.avg(F.col(c).isNull().cast("double")).alias(c) for c in features.NUMERIC]
).first()
print("null share per numeric feature (rows before 28 days of history, new launches):")
for c in features.NUMERIC:
    print(f"  {c:<20} {nulls[c]:.4f}")

{'rows': 10108926, 'store_skus': 28005, 'first_date': datetime.date(2025, 10, 1), 'last_date': datetime.date(2026, 10, 1)}
null share per numeric feature (rows before 28 days of history, new launches):
  lag_1                0.0028
  lag_7                0.0194
  lag_14               0.0388
  lag_28               0.0776
  mean_7               0.0028
  mean_28              0.0028
  std_28               0.0055
  dow_mean_4w          0.0194
  instock_rate_28      0.0035
  stockout_share_7     0.0028
  stockout_share_28    0.0028
  censored_prev_day    0.0028
  online_share_28      0.0083
  category_trend       0.0027
  dow                  0.0000
  promo_discount       0.0000
  effective_price      0.0000
  unit_price           0.0000
  perishable           0.0000
  shelf_life_days      0.0000
  lead_time_days       0.0000
  case_pack            0.0000


In [ ]:
display(
    ft.where(F.col("date") == F.lit("2026-09-30"))
    .orderBy("store_id", "sku_id")
    .select(
        *features.KEYS,
        "lag_1",
        "mean_28",
        "instock_rate_28",
        "stockout_share_28",
        "promo_discount",
        "category_trend",
        "category",
    )
    .limit(5)
)

| store_id | sku_id | date | lag_1 | mean_28 | instock_rate_28 | stockout_share_28 | promo_discount | category_trend | category |
|---|---|---|---|---|---|---|---|---|---|
| S001 | SKU0001 | 2026-09-30 | 9.0 | 10.607142857142858 | 10.607142857142858 | 0.0 | 0.0 | 0.995326218313653 | Dairy |
| S001 | SKU0002 | 2026-09-30 | 2.0 | 2.607142857142857 | 2.607142857142857 | 0.0 | 0.0 | 0.995326218313653 | Dairy |
| S001 | SKU0006 | 2026-09-30 | 4.0 | 6.607142857142857 | 6.607142857142857 | 0.0 | 0.0 | 0.995326218313653 | Dairy |
| S001 | SKU0007 | 2026-09-30 | 3.0 | 2.5 | 2.5454545454545454 | 0.017857142857142856 | 0.0 | 0.995326218313653 | Dairy |
| S001 | SKU0008 | 2026-09-30 | 11.0 | 9.928571428571429 | 9.928571428571429 | 0.0 | 0.0 | 0.995326218313653 | Dairy |


In [ ]:
dbutils.notebook.exit(
    json.dumps(
        {
            "table": TABLE,
            "mode": MODE,
            "rows": summary["rows"],
            "store_skus": summary["store_skus"],
            "first_date": str(summary["first_date"]),
            "last_date": str(summary["last_date"]),
        }
    )
)

{"table": "bigbox_fe_bar.ml.store_sku_features", "mode": "full", "rows": 10108926, "store_skus": 28005, "first_date": "2025-10-01", "last_date": "2026-10-01"}